# Decoder-only Architecture: build the block, trace the shapes, prove no leakage

This notebook is the runnable companion to the **Decoder-only Architecture** concept page. We build the exact stack the page draws — token + positional embedding → *N* × [ pre-norm → causal multi-head self-attention → residual → pre-norm → MLP → residual ] → final norm → weight-tied LM head → next-token logits — on tiny dimensions (`d_model=32`, 4 heads, 2 layers) you can hold in your head, and **print the tensor shape at every step**.

The headline check is the **no-leakage test**: changing a *future* token must leave the logits at every *earlier* position bit-for-bit identical. That is the causal mask doing its one job, and it is what makes teacher-forced parallel training legal. We assert it *before* anything else.

Verified on Python 3.12 / torch 2.12.0. Device-agnostic (CUDA / MPS / CPU); seeded for reproducibility.

> **Step 1 — imports, dimensions, and the device.** Everything below runs on one tiny model. We hoist all dimensions and the attention scale into named constants so there are no magic numbers in the math, and pick the best available device (CUDA → MPS → CPU) with CPU as the reproducible fallback. The attention scale `1/sqrt(head_dim)` keeps dot products from growing with `head_dim` and saturating softmax.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Model dimensions for the tiny demo stack (small enough to trace by eye).
VOCAB_SIZE = 50          # toy vocabulary; real GPT-2 uses 50,257 sub-word tokens
D_MODEL = 32             # model width: every token is a 32-d vector through the whole stack
N_HEADS = 4              # attention heads; each works in a D_MODEL // N_HEADS = 8-d subspace
HEAD_DIM = D_MODEL // N_HEADS    # 8; n_heads * head_dim must equal d_model
N_LAYERS = 2             # depth: how many identical decoder blocks we stack
FFN_EXPANSION = 4        # MLP hidden width = 4 * d_model, the classic Transformer choice
SEQ_LEN = 8              # tokens in the demo sequence
BATCH = 1                # one sequence; the math is identical per batch element
SEED = 0                 # fixed so CPU runs are bit-for-bit reproducible
LEAK_PROBE_OFFSET = 7    # any non-zero shift to a future token for the leakage probe

ATTENTION_SCALE = HEAD_DIM ** -0.5   # 1/sqrt(head_dim): tame scores before softmax

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", DEVICE)
print("torch :", torch.__version__)

device: mps
torch : 2.12.0


> **Step 2 — the causal mask.** This is the single idea that makes a Transformer block "decoder-only". We build an *additive* mask: `0` on and below the diagonal, `-inf` strictly above it. Adding it to the raw scores **before** softmax sends every future key to `exp(-inf) = 0` weight, so query position *i* can attend only to positions ≤ *i*. We add pre-softmax (not multiply post-softmax) so the surviving allowed keys still renormalize to sum to 1 — the future gets *no* weight, not a small one.

In [2]:
def causal_mask(seq_len: int, device: str = DEVICE) -> torch.Tensor:
    """0 on/below the diagonal, -inf strictly above it (the forbidden future)."""
    # triu(..., diagonal=1) keeps the STRICTLY-upper triangle (j > i) and zeros the rest.
    return torch.triu(
        torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1
    )

# Sanity-check on a 4-token mask: equal (zero) scores isolate the masking effect.
demo_len = 4
weights = F.softmax(torch.zeros(demo_len, demo_len) + causal_mask(demo_len, "cpu"), dim=-1)
print(weights)
print("each row sums to 1 :", torch.allclose(weights.sum(-1), torch.ones(demo_len)))
print("token2 -> token3 (future):", float(weights[2, 3]))   # 0.0 — cannot see the future

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
each row sums to 1 : True
token2 -> token3 (future): 0.0


The lower-triangular staircase is exactly what the page derives by hand: row 0 sees only itself, row 1 sees tokens 0–1, row *t* sees tokens 0…*t*. The future weight is **exactly** `0.0` and every row is a valid distribution — `torch.triu(..., diagonal=1)` is the one-liner every implementation uses, so memorize it.

> **Step 3 — one decoder block.** Pre-norm causal self-attention + pre-norm MLP, each wrapped in a residual: `x = x + sublayer(norm(x))`. Pre-norm keeps a clean identity "highway" for gradients, which is why deep stacks train stably (Xiong et al. 2020). Read the attention method as: fuse Q/K/V in one matmul → split into heads → scaled `q·kᵀ` scores → add the causal mask → softmax → weighted sum of values → merge heads → output projection. The block is shape-preserving `(batch, T, d) → (batch, T, d)`, which is *why* you can stack `L` of them like Lego.

In [3]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int) -> None:
        super().__init__()
        assert n_heads * (d_model // n_heads) == d_model, "n_heads must divide d_model"
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)   # fused Q,K,V — one matmul, identical math
        self.out_proj = nn.Linear(d_model, d_model)  # W_O: mixes the heads back together
        self.norm_attn = nn.LayerNorm(d_model)       # pre-norm before attention
        self.norm_ffn = nn.LayerNorm(d_model)        # pre-norm before the MLP
        self.ffn = nn.Sequential(                    # attention COMMUNICATES; the MLP COMPUTES
            nn.Linear(d_model, FFN_EXPANSION * d_model),
            nn.GELU(),
            nn.Linear(FFN_EXPANSION * d_model, d_model),
        )

    def self_attention(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, d_model = x.shape
        q, k, v = self.qkv(x).split(d_model, dim=-1)             # each (batch, seq_len, d_model)

        def to_heads(t: torch.Tensor) -> torch.Tensor:          # -> (batch, n_heads, seq_len, head_dim)
            return t.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = to_heads(q), to_heads(k), to_heads(v)
        scores = (q @ k.transpose(-1, -2)) * ATTENTION_SCALE    # (batch, n_heads, seq_len, seq_len)
        scores = scores + causal_mask(seq_len, device=str(x.device))   # forbid the future
        weights = F.softmax(scores, dim=-1)                     # a distribution per query row
        context = weights @ v                                   # weighted sum of values
        # transpose heads beside head_dim FIRST, then reshape — else heads interleave and corrupt d_model
        merged = context.transpose(1, 2).reshape(batch, seq_len, d_model)
        return self.out_proj(merged)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attention(self.norm_attn(x))   # pre-norm attention + residual
        x = x + self.ffn(self.norm_ffn(x))               # pre-norm MLP + residual
        return x

print("DecoderBlock defined")

DecoderBlock defined


> **Step 4 — the full model.** Wrap `L` blocks between a token+positional embedding at the front and a final-norm + LM head at the back. The crucial line is **weight tying**: `self.lm_head.weight = self.token_embedding.weight` — the head literally *is* the embedding matrix. The vector that represents a token going in and the vector that scores it coming out share one space; this saves `V × d` parameters and mildly improves quality (Press & Wolf 2017). Note we *add* a learned positional vector because self-attention is permutation-invariant and would otherwise have no notion of order.

In [4]:
class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)     # E: (V, d), id -> vector
        self.position_embedding = nn.Embedding(max_seq_len, d_model) # learned absolute positions
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)    # d -> V logits
        self.lm_head.weight = self.token_embedding.weight           # weight tying: head IS the embedding

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        _, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device)
        # ADD position because self-attention is permutation-invariant (no built-in order).
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)                                            # each block keeps the shape
        return self.lm_head(self.final_norm(x))

torch.manual_seed(SEED)   # seed BEFORE init so CPU runs reproduce bit-for-bit
model = DecoderOnlyLM(VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, SEQ_LEN).to(DEVICE).eval()

total_params = sum(p.numel() for p in model.parameters())
tied = model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr()
print(f"parameters: {total_params:,}  | LM head tied to token embedding: {tied}")

parameters: 27,328  | LM head tied to token embedding: True


## Trace the forward pass: a shape at every step

> **Step 5 — watch the data flow.** Feed one length-8 sequence through and print the shape after each stage. The thing to notice: from the embedding onward every tensor is `(batch, seq_len, d_model)` and **stays that shape through every block** — shape invariance is exactly why `L` identical blocks compose. Only the final LM head changes the last dimension, projecting `d_model → vocab` to produce one logit per vocabulary token at every position.

In [5]:
torch.manual_seed(SEED)
token_ids = torch.randint(0, VOCAB_SIZE, (BATCH, SEQ_LEN), device=DEVICE)

positions = torch.arange(SEQ_LEN, device=DEVICE)
embedded = model.token_embedding(token_ids) + model.position_embedding(positions)
print(f"token ids                 : {tuple(token_ids.shape)}  (batch, seq_len)")
print(f"after embedding + position: {tuple(embedded.shape)}  (batch, seq_len, d_model)")

x = embedded
for layer_index, block in enumerate(model.blocks):
    x = block(x)
    print(f"after decoder block {layer_index}     : {tuple(x.shape)}  (shape preserved)")

logits = model.lm_head(model.final_norm(x))
print(f"next-token logits         : {tuple(logits.shape)}  (batch, seq_len, vocab)")

token ids                 : (1, 8)  (batch, seq_len)
after embedding + position: (1, 8, 32)  (batch, seq_len, d_model)
after decoder block 0     : (1, 8, 32)  (shape preserved)
after decoder block 1     : (1, 8, 32)  (shape preserved)
next-token logits         : (1, 8, 50)  (batch, seq_len, vocab)


## Prove the causal mask works: the no-leakage test

> **Step 6 — the check that matters most.** The causal mask's entire promise is that position *i*'s output depends only on positions ≤ *i*. We test it directly: take the logits, then change **only the last (future-most) token** to a different id, and re-run. If the mask is honest, the logits at *every earlier position* must be **bit-for-bit identical** — `torch.equal`, max diff exactly `0.0`. This is not a soft tolerance check; a single non-zero difference would mean the future leaked backward and teacher-forced parallel training would be cheating.

In [6]:
last_position = SEQ_LEN - 1
with torch.no_grad():
    logits_original = model(token_ids)
    perturbed = token_ids.clone()
    # Flip ONLY the last token to a different id; everything before it is untouched.
    perturbed[0, last_position] = (token_ids[0, last_position] + LEAK_PROBE_OFFSET) % VOCAB_SIZE
    logits_perturbed = model(perturbed)

prefix_original = logits_original[0, :last_position]
prefix_perturbed = logits_perturbed[0, :last_position]
max_diff = (prefix_original - prefix_perturbed).abs().max().item()
identical = torch.equal(prefix_original, prefix_perturbed)
print(f"changed the LAST token id; logits at all earlier positions identical: {identical}")
print(f"max abs diff over the unchanged prefix: {max_diff:.2e}")
assert identical, "LEAKAGE: a future token moved an earlier position's logits!"
print("PASS: a future token cannot influence an earlier position's prediction.")

changed the LAST token id; logits at all earlier positions identical: True
max abs diff over the unchanged prefix: 0.00e+00
PASS: a future token cannot influence an earlier position's prediction.


## What we saw

- A **decoder-only block** is pre-norm causal self-attention + pre-norm MLP, each in a residual — and it is **shape-preserving** `(batch, T, d) → (batch, T, d)`, which is the whole reason a stack of `L` identical blocks composes.
- The **causal mask** (`-inf` strictly above the diagonal, added before softmax) gives future keys exactly zero weight while each query row still renormalizes to sum to 1.
- **Weight tying** makes the LM head *be* the token-embedding matrix — fewer parameters, mildly better quality.
- The **no-leakage assert** passed bit-for-bit: changing a future token left every earlier position's logits untouched. That guarantee is what lets one teacher-forced forward pass legally score all `T` next-token predictions at once — the parallel-training trick the page derives.

The full architecture story — *why* decoder-only beat encoder-only (BERT) and encoder-decoder (T5), parameter counting, weight-tying's quality win, the modern Pre-RMS-RoPE-SwiGLU-GQA recipe, and prefill-vs-decode at inference — is in the concept page next to this notebook.